<a href="https://colab.research.google.com/github/jhenaodev-pixel/Backups/blob/main/simulacionMontecarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""Estimación de π mediante el método de Monte Carlo.

La idea: se generan puntos aleatorios uniformemente distribuidos dentro del
cuadrado [-1, 1] x [-1, 1]. La proporción de puntos que caen dentro del círculo
inscrito de radio 1 se aproxima a (área del círculo) / (área del cuadrado) = π/4.
Por tanto, π ≈ 4 * (puntos_dentro / puntos_totales).

Uso:
    python monte_carlo_pi.py --n 1000000
    python monte_carlo_pi.py --n 50000 --plot --seed 42

Autor: <tu nombre>
Licencia: MIT
"""

from __future__ import annotations

import argparse
import math

import numpy as np


def estimar_pi(n_puntos: int, semilla: int | None = None) -> tuple[float, np.ndarray, np.ndarray]:
    """Estima el valor de π usando `n_puntos` muestras aleatorias.

    Args:
        n_puntos: Número de puntos aleatorios a generar. Debe ser > 0.
        semilla: Semilla para el generador aleatorio (reproducibilidad).

    Returns:
        Una tupla `(pi_estimado, puntos, dentro)` donde:
            - pi_estimado: la aproximación de π.
            - puntos: array de forma (n_puntos, 2) con las coordenadas.
            - dentro: máscara booleana; True si el punto cae en el círculo.

    Raises:
        ValueError: si `n_puntos` no es un entero positivo.
    """
    if n_puntos <= 0:
        raise ValueError("n_puntos debe ser un entero positivo.")

    rng = np.random.default_rng(semilla)
    puntos = rng.uniform(-1.0, 1.0, size=(n_puntos, 2))

    # Distancia al cuadrado desde el origen (evita calcular la raíz).
    distancia_sq = puntos[:, 0] ** 2 + puntos[:, 1] ** 2
    dentro = distancia_sq <= 1.0

    pi_estimado = 4.0 * np.count_nonzero(dentro) / n_puntos
    return pi_estimado, puntos, dentro


def graficar(puntos: np.ndarray, dentro: np.ndarray, pi_estimado: float, ruta: str = "monte_carlo_pi.png") -> None:
    """Genera y guarda una visualización de los puntos y el círculo unitario."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib no está instalado. Ejecuta: pip install matplotlib")
        return

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(puntos[dentro, 0], puntos[dentro, 1], s=1, color="#2a9d8f", label="Dentro")
    ax.scatter(puntos[~dentro, 0], puntos[~dentro, 1], s=1, color="#e76f51", label="Fuera")

    circulo = plt.Circle((0, 0), 1.0, fill=False, color="black", linewidth=1.2)
    ax.add_patch(circulo)

    ax.set_aspect("equal")
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_title(f"Monte Carlo: π ≈ {pi_estimado:.5f} ({len(puntos):,} puntos)")
    ax.legend(loc="upper right", markerscale=6)

    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    print(f"Figura guardada en: {ruta}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Estima π mediante el método de Monte Carlo."
    )
    parser.add_argument(
        "-n", "--n", type=int, default=1_000_000,
        help="Número de puntos aleatorios (por defecto: 1_000_000).",
    )
    parser.add_argument(
        "-s", "--seed", type=int, default=None,
        help="Semilla para reproducibilidad (por defecto: None).",
    )
    parser.add_argument(
        "-p", "--plot", action="store_true",
        help="Genera una imagen con la visualización de los puntos.",
    )
    # parse_known_args (en lugar de parse_args) ignora argumentos ajenos,
    # como el "-f .../kernel.json" que inyectan Jupyter y Google Colab.
    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()

    pi_estimado, puntos, dentro = estimar_pi(args.n, args.seed)
    error_abs = abs(pi_estimado - math.pi)
    error_rel = error_abs / math.pi * 100

    print(f"Puntos generados : {args.n:,}")
    print(f"π estimado       : {pi_estimado:.6f}")
    print(f"π real           : {math.pi:.6f}")
    print(f"Error absoluto   : {error_abs:.6f}")
    print(f"Error relativo   : {error_rel:.4f} %")

    if args.plot:
        graficar(puntos, dentro, pi_estimado)


if __name__ == "__main__":
    main()

Puntos generados : 1,000,000
π estimado       : 3.143756
π real           : 3.141593
Error absoluto   : 0.002163
Error relativo   : 0.0689 %
